# NimbusPay Ticket Triage — free Colab T4
Run this notebook on a Colab T4 GPU. See `COLAB_INSTRUCTIONS.md` for the full rationale and submission constraints.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload nimbuspay_colab_project.zip


In [ ]:
!unzip -q nimbuspay_colab_project.zip -d /content
%cd /content/nimbuspay_colab_project
!nvidia-smi
!pip -q install -r requirements.txt


In [ ]:
# Read-only audit: detect every malformed raw assistant completion before repairing data.
!python audit_data.py
!head -6 artifacts/audit_invalid_json.csv


In [ ]:
# Reconstruct the 25 audited truncated labels, remove train/dev input leakage, and log every change.
!python clean_data.py
!head -12 artifacts/cleaning_log.csv
!python prepare_raw.py


In [ ]:
# Required baselines. Rules are allowed only for the second baseline.
!python baseline.py --output artifacts/dev_base_fixed.jsonl
!python score.py --gold data/dev.jsonl --pred artifacts/dev_base_fixed.jsonl
!python baseline.py --with-rules --output artifacts/dev_base_rules.jsonl
!python score.py --gold data/dev.jsonl --pred artifacts/dev_base_rules.jsonl


In [ ]:
# Final QLoRA adapter: 3B base model, rank 8, 4-bit NF4, assistant-only loss.
!python train.py --train artifacts/train_clean.jsonl --output artifacts/adapter_final --load-in-4bit --rank 8 --epochs 2 --learning-rate 2e-4 --max-length 2048


In [ ]:
# Development evaluation
!python predict.py --input data/dev.jsonl --adapter artifacts/adapter_final --output artifacts/dev_final_predictions.jsonl
!python score.py --gold data/dev.jsonl --pred artifacts/dev_final_predictions.jsonl


In [ ]:
# Hidden-test submission predictions
!python predict.py --input data/test_inputs.jsonl --adapter artifacts/adapter_final --output artifacts/predictions_test.jsonl


## Controlled experiments
Use the commands in `COLAB_INSTRUCTIONS.md` for raw-vs-cleaned, 16-bit-vs-4-bit, rank, and epoch experiments. Change `--output` per run and record the scorer metrics, peak GPU memory, and wall-clock time.